# Run Advanced Analysis {#ref_sherlock_run_advanced_analysis}

This example demonstrates how to connect to the Sherlock gRPC service,
import a project, and run various analyses on a project, including part
validation, natural frequency, thermal derating, and more.

## Description

Sherlock provides the ability to perform various types of analyses on a
project. This script demonstrates how to: - Connect to the Sherlock
service. - Import a project into Sherlock. - Run several types of
analyses, such as part validation, mechanical shock, harmonic vibration,
and others.


In [ ]:
import os

from ansys.api.sherlock.v0 import SherlockAnalysisService_pb2
from examples.examples_globals import get_sherlock_tutorial_path

from ansys.sherlock.core import launcher
from ansys.sherlock.core.errors import (
    SherlockImportProjectZipArchiveError,
    SherlockRunAnalysisError,
)

# Connect to Sherlock

Connect to the Sherlock service and ensure proper initialization.


In [ ]:
sherlock = launcher.connect(port=9092, timeout=10)

# Delete Project

Delete the project if it already exists.


In [ ]:
try:
    sherlock.project.delete_project("Test")
    print("Project deleted successfully.")
except Exception:
    pass

# Import Tutorial Project

Import the tutorial project zip archive from the Sherlock tutorial
directory.


In [ ]:
try:
    sherlock.project.import_project_zip_archive(
        project="Test",
        category="Demos",
        archive_file=os.path.join(get_sherlock_tutorial_path(), "Auto Relay Project.zip"),
    )
    print("Tutorial project imported successfully.")
except SherlockImportProjectZipArchiveError as e:
    print(f"Error importing project zip archive: {e}")

# Run Multiple Analyses

Run various types of analyses on the \"Main Board\" in the \"Tutorial
Project\".


In [ ]:
try:
    # Run analyses
    pth_fatigue = SherlockAnalysisService_pb2.RunAnalysisRequest.Analysis.AnalysisType.PTHFatigue
    semiconductor_wearout = (
        SherlockAnalysisService_pb2.RunAnalysisRequest.Analysis.AnalysisType.SemiconductorWearout
    )
    thermal_derating = (
        SherlockAnalysisService_pb2.RunAnalysisRequest.Analysis.AnalysisType.ThermalDerating
    )
    component_failure_mode = (
        SherlockAnalysisService_pb2.RunAnalysisRequest.Analysis.AnalysisType.ComponentFailureMode
    )

    analysis_types = [
        (pth_fatigue, [("Phase 1", ["Thermal Event"])]),
        (semiconductor_wearout, [("Phase 1", ["Thermal Event"])]),
        (thermal_derating, [("Phase 1", ["Thermal Event"])]),
        (component_failure_mode, [("Phase 1", ["Thermal Event"])]),
    ]

    for analysis_type, params in analysis_types:
        sherlock.analysis.run_analysis(
            project="Test", cca_name="Auto Relay", analyses=[(analysis_type, params)]
        )

except SherlockRunAnalysisError as e:
    print(f"Error running analysis: {e}")